# 33. Async context managers, iteration, and cancellation

**Level:** Advanced

**Learning outcome:** Manage asynchronous object lifetimes and preserve cancellation after deterministic cleanup.

**Prerequisites:** 13, 24, and 30: context managers, async tasks, and lazy iteration.

[Learning-path index](00_START_HERE_OOP_Learning_Path.ipynb) | [Previous](32_Metaclasses_and_Class_Creation.ipynb) | [Next notebook](34_Weak_References_and_Object_Lifetime.ipynb)

## How to study this notebook
1. Read the mental model and explanation before executing a code cell.
2. Predict the output, then run the cell and compare the stored result.
3. Trace constructor calls, attribute reads/writes, returned values, and exception paths.
4. Change one input, predict the difference, and keep meaningful assertions.
5. Complete the practice prompts and answer the final checkpoint aloud.

**Run independently:** every worked code cell includes its own imports, definitions, and input data. You do not need to run another notebook first. Use Python 3.11+ and a Python Jupyter kernel. Only the standard library is used. Async cells use top-level await, not asyncio.run inside Jupyter.

**Offline safety:** all network states, commands, approvals, and failures are simulations. The code does not connect to devices, use credentials, write device configuration, or need external input files. Teaching limits and command strings are not vendor specifications. Assertions are learning checks, not production input validation.

## Mental model
async with awaits __aenter__ and __aexit__. It provides structured lifetime management when entry
or exit may need asynchronous work. async for obtains an async iterator through __aiter__ and
awaits __anext__ for each value. An async iterator terminates with StopAsyncIteration, not the
synchronous StopIteration.

Cancellation is part of task control flow, not an ordinary device failure to convert into success.
When a task is cancelled, cleanup should run and cancellation should normally propagate. Put
cleanup in finally or an async context manager, and do not broadly suppress CancelledError.
Cleanup code that itself awaits can be interrupted by further cancellation; real adapters need
a bounded, reviewed cleanup policy.

These cells use top-level await because Jupyter already runs an event loop. They coordinate
with Events rather than timing-dependent sleeps, so the cancellation example proves it reached
the active resource scope before cancellation. No sockets, devices, real delays, or background
tasks survive the cells.

## Worked example 1: Async context manager closes after a body failure

**Problem:** Await resource entry/exit and keep the original error visible.

### Step-by-step explanation
1. Initialize observable state before entering the resource scope.
2. Await entry through async with and confirm the active state inside.
3. Raise a known body error to exercise the failure path.
4. Return False from exit so the error propagates after cleanup.

**Expected output:**
```text
['open', 'close']
```

**Common mistake / limitation:** An async method does not make a blocking close operation nonblocking automatically.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [1]:
class OfflineSession:
    def __init__(self):
        self.events = []
        self.open = False

    async def __aenter__(self):
        self.open = True
        self.events.append("open")
        return self

    async def __aexit__(self, exc_type, exc_value, traceback):
        self.open = False
        self.events.append("close")
        return False

session = OfflineSession()
try:
    async with session:
        assert session.open
        raise ValueError("simulated parse error")
except ValueError as error:
    assert str(error) == "simulated parse error"
else:
    raise AssertionError("body error suppressed")
assert not session.open and session.events == ["open", "close"]
print(session.events)

['open', 'close']


### Practice 1

Use a fake transport with async close and verify it is awaited exactly once.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 2: Async iterator yields a bounded replay stream

**Problem:** Implement the separate asynchronous iteration protocol.

### Step-by-step explanation
1. Wrap a stable local record snapshot in a synchronous iterator.
2. __aiter__ returns the async iterator itself.
3. __anext__ translates exhaustion to the async protocol's termination exception.
4. Consume with async for and prove this stream is one-pass.

**Expected output:**
```text
['Ethernet1/1:up', 'Ethernet1/2:down']
```

**Common mistake / limitation:** This local replay has no real I/O and no need to be asynchronous except to demonstrate the protocol.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [2]:
class ReplayStream:
    def __init__(self, records):
        self.records = iter(tuple(records))

    def __aiter__(self):
        return self

    async def __anext__(self):
        try:
            return next(self.records)
        except StopIteration:
            raise StopAsyncIteration from None

stream = ReplayStream(["Ethernet1/1:up", "Ethernet1/2:down"])
collected = [record async for record in stream]
assert collected == ["Ethernet1/1:up", "Ethernet1/2:down"]
assert [record async for record in stream] == []
print(collected)

['Ethernet1/1:up', 'Ethernet1/2:down']


### Practice 2

Create a reusable async iterable that supplies a fresh stream for each traversal.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Worked example 3: Cancellation cleans up and still propagates

**Problem:** Cancel only after a task proves it entered its active resource scope.

### Step-by-step explanation
1. Use Events to coordinate the exact task phase instead of hoping a sleep is long enough.
2. The worker establishes state and announces that it is ready.
3. Cancel the owned task and await its completion.
4. Check that finally ran and that CancelledError was not converted into successful completion.

**Expected output:**
```text
cancellation propagated
resource cleaned: True
```

**Common mistake / limitation:** Do not leave a cancelled task unawaited; its completion is part of owning its lifetime.

**Verification:** the next code cell asserts results, state, or expected rejection. Compare those checks with the explanation, not just the printed text.

In [3]:
import asyncio

class PollWorker:
    def __init__(self):
        self.started = asyncio.Event()
        self.release = asyncio.Event()
        self.open = False
        self.cleaned = False

    async def run(self):
        self.open = True
        try:
            self.started.set()
            await self.release.wait()
        finally:
            self.open = False
            self.cleaned = True

worker = PollWorker()
task = asyncio.create_task(worker.run())
await worker.started.wait()
assert worker.open
task.cancel()
try:
    await task
except asyncio.CancelledError:
    print("cancellation propagated")
else:
    raise AssertionError("cancellation swallowed")
assert worker.cleaned and not worker.open and task.cancelled()
print("resource cleaned:", worker.cleaned)

cancellation propagated
resource cleaned: True


### Practice 3

Replace the state flags with an injected async context manager and keep the same assertions.

Predict the result before executing your extension. Keep one normal case and one edge/rejection case. State whether the extension changes identity, ownership, input validation, or error propagation.

## Check your understanding: Async context managers, iteration, and cancellation

**You should now be able to:** Manage asynchronous object lifetimes and preserve cancellation after deterministic cleanup.

1. Explain which object owns each piece of state and which operations can change it.
2. Walk through a normal result and an invalid-input or failure path without running the code.
3. Name one invariant, one useful assertion, and one production assumption not proved by the fake.
4. Implement one extension prompt, including a test that would fail if your new behavior were wrong.
5. Explain when a function or a dataclass would be simpler than another subclass or framework.

**Do not confuse:** a successful local assertion with real device compatibility; missing data with a healthy result; underscore naming with access control; type annotations with runtime validation; or local rollback with a multi-device transaction.

## Official reference reading
- `https://docs.python.org/3.13/reference/datamodel.html`
- `https://docs.python.org/3.13/library/asyncio-task.html`
- `https://docs.python.org/3.13/library/asyncio-sync.html`

References describe Python contracts. They do not certify any vendor command or network policy.